# Fine-tune a small VLM to classify medical bill pagesTeaches Qwen2-VL-2B to look at a bill page and say its type:Bill Summary, Bill Detail, Pharmacy Bill, Lab Bill, or Other.**Runtime → Change runtime type → T4 GPU** before you start.---### Important: this notebook is safe to re-runEvery cell can be run again without breaking anything. Labels are saved to disk after**every single page**, so you never lose your work — even if Colab disconnects or youaccidentally re-run an earlier cell.If you get disconnected, just run the cells from the top again. Labelling will continuefrom where you stopped.

## 1. Check the GPU

In [ ]:
import torchif not torch.cuda.is_available():    print("NO GPU FOUND")    print("Go to: Runtime -> Change runtime type -> T4 GPU, then run this cell again")else:    print("GPU:", torch.cuda.get_device_name(0))    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")    # A T4 is an older GPU. It cannot do bfloat16 maths, only float16.    # Using the wrong one crashes training with a confusing dtype error.    if torch.cuda.get_device_capability()[0] >= 8:        print("This GPU supports bfloat16")    else:        print("Older GPU (T4) -> the notebook will use float16 automatically")

## 2. Install libraries (about 2 minutes)

In [ ]:
!pip -q install "transformers>=4.49.0" "peft>=0.13.0" "bitsandbytes>=0.44.0" accelerate pymupdfprint("Done")

## 3. SettingsAll the choices are in one place so you never have to hunt for them.

In [ ]:
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"LABEL_OPTIONS = ["Bill Summary", "Bill Detail", "Pharmacy Bill", "Lab Bill", "Other"]LABELS_FILE = "my_labels.json"     # your labels are saved here after every pageQUESTION = (    "Look at this medical bill page and tell me its type.\n"    "Choose exactly one from this list:\n"    + "\n".join(f"- {label}" for label in LABEL_OPTIONS)    + "\n\nAnswer with only the type name, nothing else.")# Two of the sample files are actually the SAME 90-page bill split into two parts.# We treat them as one document so they never end up on opposite sides of the split.SAME_DOCUMENT = {"train_sample_9": "bill_A", "train_sample_10": "bill_A"}def pick_dtype():    """Work out float16 vs bfloat16 from the GPU we actually have.    This is a function, not a variable, so that any cell can call it at any time.    Previously it was a variable set in one cell and used in another - which broke    if the runtime restarted or cells were run out of order.    """    import torch    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8    return use_bf16, (torch.bfloat16 if use_bf16 else torch.float16)use_bf16, compute_dtype = pick_dtype()print("Will train in", "bfloat16" if use_bf16 else "float16")

## 4. Upload your PDFsChoose all 15 `train_sample_*.pdf` files.If you already uploaded them in this session, this cell will skip the upload.

In [ ]:
import osos.makedirs("pdfs", exist_ok=True)existing = [f for f in os.listdir("pdfs") if f.endswith(".pdf")]if existing:    print(f"Already have {len(existing)} PDFs. Skipping upload.")    print("(If you want to upload different files, delete the 'pdfs' folder first.)")else:    from google.colab import files    uploaded = files.upload()    for filename in uploaded:        os.rename(filename, f"pdfs/{filename}")    print("Uploaded", len(uploaded), "files")

## 5. Turn PDF pages into imagesThe model looks at pictures, not PDFs.**This cell is now safe to re-run.** It keeps any labels you have already typed, instead ofwiping them. That was a bug in the earlier version of this notebook.

In [ ]:
import fitz          # PyMuPDFimport jsonimport osos.makedirs("pages", exist_ok=True)# Load any labels we saved earlier, so re-running this cell does NOT lose them.saved_labels = {}if os.path.exists(LABELS_FILE):    with open(LABELS_FILE) as f:        for entry in json.load(f):            if entry.get("label"):                saved_labels[entry["image_path"]] = entry["label"]    print(f"Found {len(saved_labels)} labels already saved - keeping them")all_pages = []for pdf_name in sorted(os.listdir("pdfs")):    if not pdf_name.endswith(".pdf"):        continue    doc_name = pdf_name.replace(".pdf", "")    document = fitz.open(f"pdfs/{pdf_name}")    for page_number in range(len(document)):        image_path = f"pages/{doc_name}_page{page_number + 1}.png"        # Only render if the image is not already there (saves time on re-runs)        if not os.path.exists(image_path):            page = document[page_number]            page.get_pixmap(matrix=fitz.Matrix(2, 2)).save(image_path)        entry = {            "image_path": image_path,            "document": doc_name,            "page_number": page_number + 1,        }        if image_path in saved_labels:            entry["label"] = saved_labels[image_path]        all_pages.append(entry)    document.close()labelled_so_far = sum(1 for p in all_pages if "label" in p)print(f"{len(all_pages)} page images ready")print(f"{labelled_so_far} already labelled, {len(all_pages) - labelled_so_far} still to do")

## 6. Label the pagesTakes about 90 minutes for 50 pages. This is the slow part, and it is worth doing properly.You could ask Gemini to label them instead. But then your accuracy score would only measure*"does my model agree with Gemini"*, not *"is my model correct"* — and your model could neverscore above Gemini, because Gemini's answers would be the answer key.**Your work is saved after every single page.** If Colab disconnects, run cells 1–5 again andthen this cell; it will continue from where you stopped.Type `stop` at any time to pause and come back later.

In [ ]:
from IPython.display import display, Image as ShowImageimport jsondef save_labels():    with open(LABELS_FILE, "w") as f:        json.dump(all_pages, f, indent=2)print("1 = Bill Summary   only category totals, no individual items")print("2 = Bill Detail    a table of individual services with amounts")print("3 = Pharmacy Bill  a medicine / drug bill")print("4 = Lab Bill       a pathology or diagnostic test bill")print("5 = Other          anything else (implant invoice, cover page)")print()print("Type 'stop' to pause. Your progress is saved after every page.")print()todo = [p for p in all_pages if "label" not in p]if not todo:    print("Everything is already labelled. Nothing to do.")else:    for position, page in enumerate(todo, start=1):        display(ShowImage(filename=page["image_path"], width=520))        print(f"[{position} of {len(todo)}]  {page['document']}  page {page['page_number']}")        answer = input("Label (1-5, or 'stop'): ").strip().lower()        if answer == "stop":            save_labels()            print(f"\nPaused. {sum(1 for p in all_pages if 'label' in p)} pages done so far.")            print("Run this cell again whenever you want to continue.")            break        if answer in ["1", "2", "3", "4", "5"]:            page["label"] = LABEL_OPTIONS[int(answer) - 1]        else:            matched = [l for l in LABEL_OPTIONS if l.lower() == answer]            page["label"] = matched[0] if matched else "Other"            if not matched:                print("  Did not understand that - marked as Other")        save_labels()          # save immediately, every single time        print(f"  -> {page['label']}\n")    else:        print("Finished labelling every page.")

### Download your labels (do this once labelling is finished)Keep this file. If you ever need to start a fresh Colab session, upload it back and you willnot have to label again.

In [ ]:
from collections import Counterlabelled = [p for p in all_pages if "label" in p]print(f"{len(labelled)} of {len(all_pages)} pages labelled")if len(labelled) < len(all_pages):    print("\nNot finished yet. Go back and run the labelling cell again.")else:    counts = Counter(p["label"] for p in labelled)    print("\nHow many of each type:")    for label, n in counts.most_common():        print(f"  {label:<15} {n}")    # If most pages share one label, a model that always guesses that label scores well    # without learning anything. We must beat this number for training to mean anything.    top_label, top_count = counts.most_common(1)[0]    baseline_accuracy = top_count / len(labelled)    print(f"\nAlways guessing '{top_label}' would be right {baseline_accuracy:.0%} of the time.")    print("This is the number your trained model must beat.")    from google.colab import files    files.download(LABELS_FILE)

## 7. Split into train and testWe split by **document**, not by page.Pages from one bill share the same template, printer and scanner. If page 1 is in training andpage 2 is in testing, the model has effectively already seen the test page — the score wouldlook good and mean nothing.

In [ ]:
import randomlabelled = [p for p in all_pages if "label" in p]assert labelled, "No labels found. Go back and run the labelling cell first."def document_group(doc_name):    return SAME_DOCUMENT.get(doc_name, doc_name)groups = sorted(set(document_group(p["document"]) for p in labelled))random.Random(42).shuffle(groups)      # fixed seed = same split every runn_test_groups = max(1, round(len(groups) * 0.3))test_groups = set(groups[:n_test_groups])train_pages = [p for p in labelled if document_group(p["document"]) not in test_groups]test_pages  = [p for p in labelled if document_group(p["document"]) in test_groups]from collections import Countercounts = Counter(p["label"] for p in labelled)top_label, top_count = counts.most_common(1)[0]baseline_accuracy = top_count / len(labelled)print(f"Train: {len(train_pages)} pages from {len(groups) - n_test_groups} documents")print(f"Test:  {len(test_pages)} pages from {n_test_groups} documents")print(f"\nAlways-guess baseline: {baseline_accuracy:.0%} (always saying '{top_label}')")print(f"\nNOTE: with only {len(test_pages)} test pages the accuracy is rough.")print("If you score 80%, the true value could be anywhere from about 60% to 95%.")print("Say that honestly rather than quoting a precise-sounding number.")

## 8. Load the modelLoaded in **4-bit**: each number is squeezed into 4 bits instead of 16, so the model uses abouta quarter of the memory and fits comfortably on a T4.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfigimport torchuse_bf16, compute_dtype = pick_dtype()     # recomputed here so this cell stands alonequantization_settings = BitsAndBytesConfig(    load_in_4bit=True,    bnb_4bit_quant_type="nf4",    bnb_4bit_compute_dtype=compute_dtype,    bnb_4bit_use_double_quant=True,)model = Qwen2VLForConditionalGeneration.from_pretrained(    MODEL_NAME,    quantization_config=quantization_settings,    torch_dtype=compute_dtype,    attn_implementation="sdpa",    # a T4 is too old for FlashAttention-2    device_map="auto",)# max_pixels controls image resolution. Higher lets the model read smaller text# but uses more memory. This setting is a reasonable middle ground for documents.processor = AutoProcessor.from_pretrained(    MODEL_NAME,    min_pixels=256 * 28 * 28,    max_pixels=1024 * 28 * 28,)processor.tokenizer.padding_side = "right"print("Model loaded in 4-bit")

## 9. Test the model BEFORE training**Do not skip this.** Without it, a score of 75% after training tells you nothing — the modelmight already have scored 75% before you touched it.Takes a few minutes.

In [ ]:
from PIL import Imageimport torchdef ask_model(image_path):    """Show one page to the model and get back one of our five labels."""    image = Image.open(image_path).convert("RGB")    conversation = [{"role": "user",                     "content": [{"type": "image"}, {"type": "text", "text": QUESTION}]}]    prompt_text = processor.apply_chat_template(        conversation, tokenize=False, add_generation_prompt=True)    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to(model.device)    with torch.no_grad():        output = model.generate(**inputs, max_new_tokens=12, do_sample=False)    generated = output[:, inputs.input_ids.shape[1]:]    answer = processor.batch_decode(generated, skip_special_tokens=True)[0].strip().lower()    for label in LABEL_OPTIONS:        if label.lower() in answer:            return label    return "Other"def measure(pages, description):    predictions = [ask_model(p["image_path"]) for p in pages]    correct = sum(1 for p, guess in zip(pages, predictions) if p["label"] == guess)    accuracy = correct / len(pages)    print(f"{description}: {correct}/{len(pages)} = {accuracy:.0%}")    return accuracy, predictionsprint("Measuring the model before training...\n")accuracy_before, predictions_before = measure(test_pages, "BEFORE training")

## 10. Add LoRAQwen2-VL-2B has about 2 billion numbers. Training all of them needs far more memory than a T4 has.**LoRA** freezes the original numbers and adds a small set of new ones alongside — usually under1% of the total. We train only those. That is why it fits on a free GPU, and why the saved resultis a ~40 MB file rather than 4 GB.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_trainingmodel = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)model.enable_input_require_grads()lora_settings = LoraConfig(    r=16,                # size of the added matrices. Bigger = more capacity, more memory    lora_alpha=32,       # scaling factor, usually 2x r    lora_dropout=0.05,    bias="none",    task_type="CAUSAL_LM",    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",                    "gate_proj", "up_proj", "down_proj"],)model = get_peft_model(model, lora_settings)model.print_trainable_parameters()      # see how few numbers we actually train

## 11. Train- **batch size 1 with accumulation 4** — one page at a time is all the memory allows, but we wait  for 4 pages before updating. The effect is like a batch of 4.- **learning rate 1e-4** — how big a step to take. Too high is unstable, too low changes nothing.- **float16** — because a T4 cannot do bfloat16.With ~35 pages this takes a few minutes. Watch the loss go down.

In [ ]:
from torch.utils.data import Datasetfrom transformers import Trainer, TrainingArgumentsfrom PIL import Imageuse_bf16, _ = pick_dtype()      # recomputed here, so this cell does not depend on earlier cellsclass PageDataset(Dataset):    def __init__(self, pages):        self.pages = pages    def __len__(self):        return len(self.pages)    def __getitem__(self, i):        return self.pages[i]def collate(batch):    images, full_texts, prompt_lengths = [], [], []    for page in batch:        image = Image.open(page["image_path"]).convert("RGB")        images.append(image)        full = [            {"role": "user", "content": [{"type": "image"},                                         {"type": "text", "text": QUESTION}]},            {"role": "assistant", "content": [{"type": "text", "text": page["label"]}]},        ]        full_texts.append(processor.apply_chat_template(full, tokenize=False))        # Length of the question part, so we know where the answer starts        question_text = processor.apply_chat_template(            [full[0]], tokenize=False, add_generation_prompt=True)        q_ids = processor(text=[question_text], images=[image], return_tensors="pt").input_ids        prompt_lengths.append(q_ids.shape[1])    inputs = processor(text=full_texts, images=images, return_tensors="pt", padding=True)    # -100 means "do not learn from this position".    # We mask the question so the model only learns to produce the ANSWER,    # not to repeat our own question back at us.    labels = inputs.input_ids.clone()    labels[inputs.attention_mask == 0] = -100    for i, length in enumerate(prompt_lengths):        labels[i, :length] = -100    inputs["labels"] = labels    return inputstrainer = Trainer(    model=model,    args=TrainingArguments(        output_dir="my_adapter",        num_train_epochs=8,        per_device_train_batch_size=1,        gradient_accumulation_steps=4,        learning_rate=1e-4,        lr_scheduler_type="cosine",        warmup_ratio=0.05,        logging_steps=5,        save_strategy="no",        fp16=not use_bf16,        bf16=use_bf16,        gradient_checkpointing=True,        gradient_checkpointing_kwargs={"use_reentrant": False},        remove_unused_columns=False,     # our collator needs the raw dictionaries        report_to=[],    ),    train_dataset=PageDataset(train_pages),    data_collator=collate,)trainer.train()print("\nTraining finished")

## 12. Measure again and compareThree numbers matter. Training only helped if the third clearly beats the first two.

In [ ]:
model.eval()print("Measuring the model after training...\n")accuracy_after, predictions_after = measure(test_pages, "AFTER training")print("\n" + "=" * 52)print("RESULTS")print("=" * 52)print(f"Always guess most common label : {baseline_accuracy:.0%}")print(f"Model before training          : {accuracy_before:.0%}")print(f"Model after training           : {accuracy_after:.0%}")print(f"Test set size                  : {len(test_pages)} pages")print()if accuracy_after > max(accuracy_before, baseline_accuracy):    print("Training helped - the model beats both its starting point and the baseline.")elif accuracy_after <= baseline_accuracy:    print("Training did NOT beat the always-guess baseline.")    print("This is a real result, not something to hide. Likely reasons:")    print("  - very little data (about 35 training pages)")    print("  - some pages genuinely fit two categories, so the labels are ambiguous")else:    print("No improvement over the starting point. You could try more epochs,")    print("or check whether your labels are consistent.")print("\nPages it got wrong:")for page, guess in zip(test_pages, predictions_after):    if guess != page["label"]:        print(f"  {page['document']} page {page['page_number']}: "              f"said '{guess}', correct answer '{page['label']}'")

## 13. Save the adapter

In [ ]:
model.save_pretrained("my_adapter")!du -sh my_adapter!tar czf my_adapter.tar.gz my_adapterfrom google.colab import filesfiles.download("my_adapter.tar.gz")

## What to say in the interviewDo not say *"I fine-tuned a VLM and it works."* Say what you measured:> "I fine-tuned Qwen2-VL-2B with LoRA to classify medical bill pages into five types.> I hand-labelled the pages myself rather than using another model to label them, because> otherwise I would only be measuring agreement with that model, not accuracy.>> I split by document rather than by page, since pages from one bill share a template and a> page-level split would leak information into the test set.>> I measured three things: an always-guess-the-most-common-class baseline, the base model> before training, and the model after training. [your three numbers]>> The test set is only about 15 pages, so the confidence interval is wide. I would call this a> pilot result rather than a reliable accuracy figure."That last paragraph is what will stand out. Most candidates quote a number; very few say howmuch to trust it.**If the result was bad, say so.** *"Training did not beat the baseline, and I think 35 pages istoo few for five classes"* is a stronger answer than a number you cannot defend. The JD says*measure everything* — it does not say every experiment must succeed.